# Notebook 4: Convolutional Neural Network (CNN)

**Goal:** Build, train, and evaluate a CNN for CIFAR-10 image classification.

This is the core notebook where deep learning happens! We'll:
1. Set up data with preprocessing and augmentation
2. Build our CNN architecture (layer by layer explanation)
3. Train the model for 30 epochs
4. Visualize the learning process (loss and accuracy curves)
5. Evaluate on the test set (confusion matrix, per-class accuracy)
6. Visualize what the CNN learned (filters and feature maps)

---

## 4.1 Setup and Data Preprocessing

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# Our custom modules
from src.data_utils import get_cifar10_loaders, get_raw_cifar10, CIFAR10_CLASSES, CIFAR10_MEAN, CIFAR10_STD
from src.models import SimpleCNN, count_parameters
from src.train import train_model, evaluate
from src.visualize import (plot_training_curves, plot_confusion_matrix, 
                           plot_misclassified, plot_per_class_accuracy, visualize_filters)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Check if GPU is available (CPU works fine too, just slower)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. Training will run on CPU (~20-40 minutes).")
    print("This is perfectly fine for CIFAR-10!")

Using device: cpu
No GPU detected. Training will run on CPU (~20-40 minutes).
This is perfectly fine for CIFAR-10!


### Data Augmentation and Normalization

Before feeding images to the CNN, we apply **transforms**:

1. **Data Augmentation** (training only):
   - `RandomHorizontalFlip()`: 50% chance of flipping left↔right. A flipped cat is still a cat!
   - `RandomCrop(32, padding=4)`: Pad with 4 pixels, then randomly crop back to 32x32. Simulates objects at slightly different positions.

2. **Normalization** (both training and testing):
   - `ToTensor()`: Convert pixel values from [0, 255] to [0.0, 1.0]
   - `Normalize(mean, std)`: Subtract mean, divide by std → values centered around 0

**Why augmentation helps:** It artificially increases the diversity of training data. The model sees slightly different versions of each image every epoch, forcing it to learn general features rather than memorizing specific images.

In [2]:
# Create data loaders WITH augmentation for training, WITHOUT for testing
# get_cifar10_loaders() handles all the transform setup internally
# See src/data_utils.py for the full transform pipeline

train_loader, test_loader = get_cifar10_loaders(
    batch_size=64,       # Process 64 images at a time
    data_dir='../data',
    augment=True,        # Apply random flips and crops to training data
    num_workers=0        # Number of parallel data loading processes
)

print(f"Training batches: {len(train_loader)} (each with {train_loader.batch_size} images)")
print(f"Test batches: {len(test_loader)}")
print(f"Total training images: {len(train_loader.dataset):,}")
print(f"Total test images: {len(test_loader.dataset):,}")

# Let's peek at one batch to understand the data shape
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}")
print(f"  → {images.shape[0]} images per batch")
print(f"  → {images.shape[1]} color channels (R, G, B)")
print(f"  → {images.shape[2]}x{images.shape[3]} pixels")
print(f"Labels shape: {labels.shape}")
print(f"Sample labels: {labels[:10].tolist()}")
print(f"  = {[CIFAR10_CLASSES[l] for l in labels[:10].tolist()]}")

Training batches: 782 (each with 64 images)
Test batches: 157
Total training images: 50,000
Total test images: 10,000

Batch shape: torch.Size([64, 3, 32, 32])
  → 64 images per batch
  → 3 color channels (R, G, B)
  → 32x32 pixels
Labels shape: torch.Size([64])
Sample labels: [7, 4, 8, 4, 1, 9, 6, 0, 2, 3]
  = ['horse', 'deer', 'ship', 'deer', 'automobile', 'truck', 'frog', 'airplane', 'bird', 'cat']


c:\Users\Prithvi Nair\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# Let's visualize the effect of data augmentation
# We'll show the same image with different random augmentations

# Load one raw image (no transforms)
raw_train, _ = get_raw_cifar10(data_dir='../data')
original_img, original_label = raw_train[0]

# Define the augmentation transform
augment_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
])

fig, axes = plt.subplots(1, 6, figsize=(14, 2.5))
axes[0].imshow(original_img)
axes[0].set_title('Original', fontsize=10)
axes[0].axis('off')

for i in range(5):
    augmented = augment_transform(original_img)
    axes[i+1].imshow(augmented)
    axes[i+1].set_title(f'Augmented {i+1}', fontsize=10)
    axes[i+1].axis('off')

plt.suptitle(f'Data Augmentation: Same {CIFAR10_CLASSES[original_label]}, Different Variations',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4.2 Build the CNN Architecture

Now let's build our CNN! Here's the architecture at a glance:

```
INPUT: 3x32x32 image
    ↓
BLOCK 1: Conv(3→32) → BN → ReLU → Conv(32→32) → BN → ReLU → MaxPool → Dropout
    ↓ (now 32x16x16)
BLOCK 2: Conv(32→64) → BN → ReLU → Conv(64→64) → BN → ReLU → MaxPool → Dropout
    ↓ (now 64x8x8)
FLATTEN: 64×8×8 = 4096
    ↓
CLASSIFIER: Linear(4096→512) → ReLU → Dropout → Linear(512→10)
    ↓
OUTPUT: 10 class scores
```

In [ ]:
# Create our CNN model
model = SimpleCNN(num_classes=10)

print("Model Architecture:")
print("=" * 60)
print(model)
print("=" * 60)

n_params = count_parameters(model)
print(f"\nTotal trainable parameters: {n_params:,}")

In [ ]:
# Trace the data flow through the network step by step

print("Data flow through the CNN:")
print("=" * 60)

dummy_input = torch.randn(1, 3, 32, 32)
print(f"Input:           {list(dummy_input.shape)}  (batch=1, channels=3, H=32, W=32)")

x = dummy_input
for name, layer in model.features.named_children():
    x = layer(x)
    layer_type = layer.__class__.__name__
    print(f"  After {layer_type:20s}: {list(x.shape)}")

x = x.view(x.size(0), -1)
print(f"  After Flatten:             {list(x.shape)}  (batch=1, features=4096)")

for name, layer in model.classifier.named_children():
    x = layer(x)
    layer_type = layer.__class__.__name__
    print(f"  After {layer_type:20s}: {list(x.shape)}")

print(f"\nOutput:          {list(x.shape)}  (batch=1, classes=10)")

## 4.3 Train the Model

Now the exciting part — training! Here's what happens during training:

1. **Forward pass:** Feed a batch of 64 images through the CNN → get 64 sets of 10 scores
2. **Compute loss:** Compare predictions to true labels using CrossEntropyLoss
3. **Backward pass (backpropagation):** Compute how each parameter should change to reduce the loss
4. **Update parameters:** Adjust weights using the Adam optimizer
5. **Repeat** for all 782 batches = one **epoch**
6. **Repeat** for 30 epochs

We'll track training loss, training accuracy, test loss, and test accuracy after each epoch.

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=30,
    lr=0.001,
    device=device
)

In [7]:
# Save the trained model so we don't have to retrain it
# torch.save() saves all model parameters to a file
# The file is a dictionary containing all weight tensors

torch.save(model.state_dict(), '../saved_models/simple_cnn.pth')
print("Model saved to saved_models/simple_cnn.pth")
print(f"File contains {n_params:,} learned parameter values.")

Model saved to saved_models/simple_cnn.pth
File contains 2,168,746 learned parameter values.


## 4.4 Training Visualization

Let's plot the training curves — these are the most important diagnostic tool for understanding how training went.

In [ ]:
# Plot training and validation loss/accuracy curves
fig = plot_training_curves(history)
plt.show()

## 4.5 Test Set Evaluation

Now let's do a thorough evaluation on the test set — the 10,000 images the model has NEVER seen during training.

In [ ]:
# Evaluate the final model on the test set
criterion = nn.CrossEntropyLoss()
test_loss, test_acc, predictions, true_labels = evaluate(
    model, test_loader, criterion, device
)

print(f"\nFinal Test Results:")
print(f"  Test Loss:     {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.2f}%")

In [ ]:
# Confusion Matrix
fig = plot_confusion_matrix(
    true_labels, predictions, CIFAR10_CLASSES,
    title=f'CNN Confusion Matrix (Test Accuracy: {test_acc:.1f}%)'
)
plt.show()

In [ ]:
# Per-class accuracy
fig = plot_per_class_accuracy(true_labels, predictions, CIFAR10_CLASSES)
plt.show()

In [ ]:
# Misclassified examples
_, raw_test = get_raw_cifar10(data_dir='../data')
test_images = [np.array(raw_test[i][0]) for i in range(len(raw_test))]

fig = plot_misclassified(
    test_images, true_labels, predictions, CIFAR10_CLASSES, n=20
)
plt.show()

## 4.6 What Did the CNN Learn?

Let's peek inside the trained CNN to see what patterns it learned.

In [ ]:
# Visualize the first convolutional layer's learned filters
fig = visualize_filters(model)
plt.show()

In [ ]:
# Visualize feature maps: what the CNN "sees" when processing an image

sample_idx = 0
sample_image = test_images[sample_idx]
sample_label = true_labels[sample_idx]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)
])

from PIL import Image
img_pil = Image.fromarray(sample_image)
img_tensor = transform(img_pil).unsqueeze(0).to(device)

# Extract feature maps from each conv block
model.eval()
feature_maps = []
x = img_tensor

for i, layer in enumerate(model.features):
    x = layer(x)
    if isinstance(layer, nn.ReLU):
        feature_maps.append(x.detach().cpu())

# Plot feature maps from the first conv block
fig, axes = plt.subplots(4, 9, figsize=(14, 6))

axes[0, 0].imshow(sample_image)
axes[0, 0].set_title(f'Original\n({CIFAR10_CLASSES[sample_label]})', fontsize=8)
axes[0, 0].axis('off')

fm = feature_maps[0].squeeze()
idx = 1
for row in range(4):
    start = 1 if row == 0 else 0
    for col in range(start, 9):
        if idx - 1 < fm.shape[0]:
            axes[row, col].imshow(fm[idx-1].numpy(), cmap='viridis')
            axes[row, col].set_title(f'F{idx}', fontsize=6)
        axes[row, col].axis('off')
        idx += 1

plt.suptitle('Feature Maps: What the CNN "Sees" After Block 1',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Save CNN results for comparison in Notebook 5
import json

cnn_results = {
    'accuracy': test_acc,
    'total_time': sum(history['epoch_times']),
    'epochs': len(history['train_loss']),
    'predictions': predictions,
    'true_labels': true_labels,
    'history': {
        'train_loss': history['train_loss'],
        'train_acc': history['train_acc'],
        'test_loss': history['test_loss'],
        'test_acc': history['test_acc']
    },
    'n_parameters': n_params
}
# add precision
with open('../saved_models/cnn_results.json', 'w') as f:
    json.dump(cnn_results, f)

print("CNN results saved to saved_models/cnn_results.json")
print(f"\nFinal test accuracy: {test_acc:.2f}%")
print(f"Total training time: {sum(history['epoch_times']):.0f}s ({sum(history['epoch_times'])/60:.1f} min)")

CNN results saved to saved_models/cnn_results.json

Final test accuracy: 80.34%
Total training time: 2327s (38.8 min)


## 4.7 Summary

### What we accomplished:
- Built a CNN with ~2.2M parameters organized into 2 conv blocks + classifier
- Used data augmentation (flips, crops) to improve generalization
- Trained for 30 epochs using the Adam optimizer
- Used a learning rate scheduler to fine-tune in later epochs

### Key takeaways:
1. **CNNs preserve spatial structure** — they process images as 2D grids, not flat vectors
2. **Convolution is the key operation** — small filters slide across the image detecting local patterns
3. **Hierarchical feature learning** — early layers learn edges, later layers learn shapes and objects
4. **Data augmentation helps** — random transformations prevent overfitting
5. **The model learned on its own** — no human specified what features to look for

### Next: Comparing all models side by side!